# DATASCI 267 - GenAI - Assignment 5

### Overview

In Assignment 5 you will create and test a RAG system yourself as a proof of concept and then write a corresponding business report on your findings.

The overall scenario is as follows:

You work at a tech company that is looking for new ways to organize their question answering and search capabilities to accelerate both engineering activity and the marketing team's production.  Founded in 2018, your company has grown rapidly to 300 engineers across distributed teams in San Francisco, Austin, and remotely, plus a 40-person marketing organization split between demand generation, content, product marketing, and customer education. The company also wants to roll out new GenAI-based products, so a lot of the questions will center around Generative AI concepts. Product releases are done quarterly.

As a Senior ML Engineer on your company's newly formed AI Infrastructure team, you've been tapped to lead a four-week proof-of-concept (mini-POC) evaluating whether a Retrieval-Augmented Generation (RAG) system could address the challenges and support both the engineering and marketing organizations. Your executive sponsor is the VP of Engineering, who has secured buy-in from the CMO and allocated a small budget for this exploratory work.

Your POC needs to demonstrate:

**Engineering use case**: Can RAG effectively answer technical questions about GenAI concepts, internal system architecture, and implementation details by retrieving from diverse documentation sources?

**Marketing use case**: Can RAG help marketing staff quickly find accurate, approved messaging about GenAI features, competitive positioning, and technical capabilities to accelerate content production?

**Practical viability**: What are the implementation complexities, accuracy levels, and user experience considerations? Is this worth a larger investment?

Success will be measured by demonstrated accuracy compared to ground truth answers and a clear recommendation on whether to proceed with a full implementation.  If recommended, then a qualitative test will be conducted to get feedback from a small pilot group (5 engineers, 3 marketers).


You will receive a gold dataset with 'good' responses to questions from marketing and engineering teams. You need to develop metric(s) that help you to evaluate how well your RAG system performs relative to the gold data. You should work with the tunable hyperparameters of the setup (LLM, chunking, embeddings, ...) for your iterations.

You will also need to write up your findings as a short 4-5 page proposal.

(See instructions throughout this notebook.)

So overall, the goals of this assignment are for you to:

*  Implement a RAG system using LangChain
*  Be able to formulate metric(s) that you may want to choose as your evaluation of the degree to which your system replicates gold answers (labeled data) that we will provide.
* Try out various hyper-parameters, settings, and other techniques to see which configuration works the best (given your chosen metric)  
* Write a comprehensive evaluation report, which also includes risks and limitations (and a lot more)

The notebook is organized as follows:

1. Set-Up

2. Base RAG components

    We will provide a base LangChain-based framework for you to use as inspiration for your RAG system. The components we’ll need include:  

  - 2.1 Text Embeddings    
  - 2.2 Text Chunking   
  - 2.3 The Vector DB & Semantic Search  
  - 2.4 The Language Model   
  - 2.5 Testing the LLM in a LangChain Chain   
  - 2.6. Setting up a simple RAG Chain     


3. Using RAG  

    We will provide a collection of documents for you to use for your RAG system. We will also provide a test of questions with "Gold" answers.  

  - 3.1 Loading of Data  
  - 3.2 Test Queries
  - 3.3 Running Your Implemented Model(s) and Your Experiments


4.  Evaluations

    Here, you will develop your strategy and conduct your evaluations of your RAG system

  - 4.1 Metrics
  - 4.2 Best Runs with Metrics


5. Final Results

    In this section, you will answer some questions and provide a link to your final report

  - 5.1 Model Specifications
  - 5.2 Three Test Questions (and Results)
  - 5.3 Five Other Questions
  - 5.4 Link To Your 4-5 Page Final POC Report and Recommendation

RULES:  

* You must only use the language models we specify here for the functions we specify
* You must only use the embedding models we specify here for the functions we specify
* You must use the vector store we specify here
* You must only use the document collection we provide. And they ALL must be in your vector store.   
* Apart from the provided specifications, you are free to experiment with a wide variety of hyperparameters (chunk sizes, prompts, etc.), langchain components (text splitters, retrievers, etc.)


**To run this notebook** you should copy it to your personal Colab Pro Google account by uploading it into your Google Drive. From there you can open it as a Colab notebook and run it.  Note: it needs a T4 GPU to run when you are working with all our data.  You can run sections 1 and 2 without the GPU to orient yourself to how LangChain RAG systems work. You should also be able to run it in a free Colab notebook.

NOTES:
* The Open Source Model we use is not trained for safety. So unsafe answers could be returned.


===========================================================================================================

## 1. Setup

We will first install a number of libraries and import what we will need.





In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [ ]:
#pip install --upgrade jupyter_client

In [ ]:
%%capture
!pip install -q -U transformers
!pip install -q  datasets  sentencepiece #loralib
!pip install -q bitsandbytes accelerate
!pip install -q -U langchain
!pip install  -q -U langchain-core
!pip install -q -U langchain-classic
!pip install -q -U einops
!pip install -q -U faiss-gpu
#!pip install -q -U langchain_community
#!pip install -q -U chromadb bs4 qdrant-client
!pip install -q -U langchainhub
!pip install -q -U langchain-huggingface
!pip install -q -U langchain-cohere
!pip install -q -U  wikipedia
!pip install -q -U  arxiv
!pip install -q -U  pymupdf
!pip install -q -U xmltodict
!pip install -q -U cohere
!pip install -q -U langchain-qdrant  # updated instead of langchain_community
!pip install -q -U evaluate # Added evaluate library

In [ ]:
%%capture
#In case we want to know our installed transformers library version
!pip list | grep transformers
!pip list | grep accelerate
!pip list | grep langchain

In [ ]:

import torch
import os
import bs4
import json
import numpy as np
import time
import pandas as pd
import csv
import re

#import os
os.environ['USER_AGENT'] = 'MyLangChainApp/1.0'

from pprint import pprint

import locale

from transformers import AutoTokenizer , AutoModelForCausalLM
from transformers import pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline

from langchain_cohere import ChatCohere

from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser


from langchain_text_splitters import CharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_classic import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma
#from langchain_community.vectorstores import Qdrant
from langchain_qdrant import QdrantVectorStore   #  importing this  instead
from qdrant_client import QdrantClient


from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.utils.math import cosine_similarity

from langchain_community.document_loaders import ArxivLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import WikipediaLoader
from langchain_community.document_loaders import OnlinePDFLoader
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import PubMedLoader


from google.colab import userdata

In [ ]:

import langchain
print(langchain.__version__)
import langchainhub

1.1.2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
locale.getpreferredencoding = lambda: "UTF-8"

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
%%capture
!pip install -U sentence_transformers
#evaluation
#!pip install bleurt
!pip install bert-score
#!pip install jaccard_scores
!pip install rouge-score
!pip install spacy
!pip install ragas

#from bleurt import score
#import numpy as np
from bert_score import score as bert_score
import datasets
from datasets import Dataset

In [ ]:
# for cosine similarity

base_embeddings = HuggingFaceEmbeddings(model_name="multi-qa-mpnet-base-dot-v1")

/tmp/ipython-input-3394612036.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  base_embeddings = HuggingFaceEmbeddings(model_name="multi-qa-mpnet-base-dot-v1")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from sentence_transformers import SentenceTransformer


# Load model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast
# OR
#embedding_model = SentenceTransformer('all-mpnet-base-v2')  # Better quality



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')


# Define your directory
output_folder = "/content/drive/MyDrive/MIDS_267/final_project/output/"
data_folder = "/content/drive/MyDrive/MIDS_267/final_project/data/"

Mounted at /content/drive


Add your keys from the secret store (do **NOT** print them out or leave them exposed as plaintext in your notebook!):

In [ ]:
#COHERE_API_KEY = userdata.get('COHERE_API_KEY')
#CLAUDE_API_KEY = userdata.get('CLAUDE_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
HF_TOKEN = userdata.get('HF_TOKEN')
# Set the environment variable
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
#os.environ["COHERE_API_KEY"] = COHERE_API_KEY
#os.environ["CLAUDE_API_KEY"] = CLAUDE_API_KEY

# Clean LLM output
- Some responses start with "RESPONSE:" or "ANSWER:" prefixes
- Responses contain numbered lists (1., 2., 3., etc.)
- Some have "REFERENCES:" sections
- Responses contain newlines within the text
- Some have formatting markers like bold text
- Quotation marks need escaping

In [ ]:
# clean llm output
import pandas as pd
import re




def clean_llm_response(response):
    """
    Clean LLM response - MINIMAL VERSION
    """
    if pd.isna(response):
        return ""

    response = str(response)

    # 1. Remove "ANSWER:" prefix only
    response = re.sub(r'^(RESPONSE|ANSWER):\s*', '', response, flags=re.IGNORECASE)

    # 2. Remove markdown formatting
    response = re.sub(r'\*\*([^*]+)\*\*', r'\1', response)
    response = re.sub(r'\*([^*]+)\*', r'\1', response)

    # 3. Normalize whitespace
    response = ' '.join(response.split())

    # NO TRUNCATION - let RAGAS see full response!

    return response.strip()

def clean_context_for_ragas(context):
    """
    Clean context to make it compatible with RAGAS parser
    """
    if pd.isna(context):
        return ""

    context = str(context)

    # 1. Replace newlines with spaces
    context = re.sub(r'\n+', ' ', context)

    # 2. Remove extra whitespace
    context = ' '.join(context.split())



    return context.strip()

def clean_question_for_ragas(question):
    """
    Clean question to make it compatible with RAGAS parser
    """
    if pd.isna(question):
        return ""

    question = str(question)

    # Remove "Question:" prefix if present
    question = re.sub(r'^Question:\s*', '', question, flags=re.IGNORECASE)

    # Remove newlines
    question = re.sub(r'\n+', ' ', question)

    # Remove extra whitespace
    question = ' '.join(question.split())

    return question.strip()





In [ ]:
# Read the RAG LLM Output from the file f'{output_folder}rag_{experiment}_outputs.csv'

def create_rag_context_list(context_str):
  """Split context by double newlines or paragraph breaks"""
  # Split by double newline
  context_list = [c.strip() for c in context_str.split('\n\n') if c.strip()]
  return context_list


def create_rag_output_df(output_file):
  """Read the RAG LLM Output from the file f'{output_folder}rag_{experiment}_outputs.csv'"""

  #rag_research_outputs.csv
  RAG_REFERENCE_GOLD = f'{data_folder}rag_reference_gold.csv'

  # Read the CSV file
  output_df = pd.read_csv(f'{output_folder}{output_file}')

  # Display first few rows
  print(output_df.head())

  # Show column names
  print(output_df.columns)

  # Show shape
  print(f"RAG Output Shape: {output_df.shape}")

  # Load gold reference to gold_df
  gold_df = pd.read_csv(RAG_REFERENCE_GOLD)
 # print(gold_df.head())

  # Show shape
  print(f"Gold Reference Shape: {gold_df.shape}")

  #Prepare data for RAGAS
  # {'question': [],'answer': [],'contexts': [],'ground_truth': []}
  question_list = []
  answer_list = []
  contexts_list = []
  ground_truth_list = []
  #ground_truth_marketing_list = []

  for index, row in output_df.iterrows():
      question = row['question']
      response = row['cleaned_response']
      #response = clean_llm_response(str(row['response']))
      context_list =  create_rag_context_list(row['context'])
      #print(context_list)
      question_num = row['question_num']
      audience_type = row['audience_type']



      #  filter the gold_df based on 'question_num' and then get the answer
      if audience_type == 'research':
        gold_ref = gold_df[gold_df['question_num'] == question_num]['gold_answer_research'].item()
      else:
        gold_ref = gold_df[gold_df['question_num'] == question_num]['gold_answer_marketing'].item()

      question_list.append(question)
      answer_list.append(response)
      contexts_list.append(context_list)
      ground_truth_list.append(gold_ref)


  ragas_data = {
      'question': question_list,
      'answer': answer_list,
      'contexts': contexts_list,
      'ground_truth': ground_truth_list
      }


  return output_df, ground_truth_list, ragas_data



## 2. Building the Components of the Scoring System

The RAG output will be scored along different dimensions to generate an overall meaningful score for the response. In particular, the following dimensions will be considered:

- Length matching : Comparing the length of the LLM response with that of the gold answer
- Semantic Similarity with Gold Response:
  - Cosine similarity
  - SBERT

- Lexical Similarity with Gold Response:
  
  - ROUGE
  - BLEU
  - Jacquard

- SpacY score
- RAGAS




Metric comparsion

| Metric | Complexity | Speed | Semantic Understanding | Best Use Case |
|--------|-----------|-------|----------------------|---------------|
| Jaccard | Very Simple | Very Fast | None | Quick word overlap check |
| ROUGE-L | Simple | Fast | None (sequence-aware) | Content overlap + order |
| BLEU | Simple | Fast | None | N-gram precision |
| BERTScore | Complex | Medium | Yes | Semantic similarity |
| BLEURT | Very Complex | Slow | Yes | Human-like evaluation |



word_overlap versus Jaccard
| Metric | Formula | Penalizes | Best For |
|--------|---------|-----------|----------|
| Word Overlap (Recall) | intersection / gold | Missing words only | Checking if gold content is covered |
| Precision | intersection / pred | Extra words only | Checking if prediction is focused |
| Jaccard | intersection / union | Both missing & extra | Balanced similarity measure |

### 2.1 Length Matching

We will compare the number of words in the LLM response and the Gold Response







In [ ]:
def calculate_length_ratio_score(gold_ref, response):

  # Length ratio
  length_ratio_score = []
  for a_gold_ref, a_response in zip(gold_ref, response):
    g_words = len(a_gold_ref.split())
    r_words = len(a_response.split())
    length_ratio = min(g_words, r_words) / max(g_words, r_words) if max(g_words, r_words) > 0 else 0
    length_ratio_score.append(length_ratio)
    # Save length_ratio_score to file 'length_ratio_scores.csv'

  return length_ratio_score

### 2.2 ROUGE-L Score

Measures longest common subsequence between the gold reference and the LLM response.

In [ ]:
from evaluate import load

rouge = load("rouge")

def calculate_rouge_l_score(gold_ref, response):
  rouge_l_score = []
  for a_gold_ref, a_response in zip(gold_ref, response):
    rouge_score = rouge.compute(predictions=[a_response], references=[a_gold_ref], use_stemmer=True)['rougeL']
    rouge_l_score.append(rouge_score)
  return rouge_l_score

### 2.3 Bleu Score



In [ ]:
from evaluate import load

sentence_bleu = load('bleu')

def calculate_bleu_score(gold_ref, response):
  bleu_scores = []
  for a_gold_ref, a_response in zip(gold_ref, response):
    bleu_result = sentence_bleu.compute(predictions=[a_response], references=[a_gold_ref])
    bleu_scores.append(bleu_result['bleu']) # Extract the actual BLEU score
  return bleu_scores

### 2.4 BERTScore
BERTScore is lighter and faster than BLEURT

In [ ]:
def calculate_bert_score(gold_ref, responses):
    """
    Evaluate using BERTScore (faster alternative to BLEURT)
    """
    P, R, F1 = bert_score(
        responses,
        gold_ref,
        lang="en",
        model_type="microsoft/deberta-xlarge-mnli",  # or "roberta-large"
        verbose=False
    )
    return {
        "precision": P.numpy(),
        "recall": R.numpy(),
        "f1": F1.numpy()
    }


### 2.5 Jaccard
Jaccard = (intersection) / (union) of word sets

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
STOPWORDS = nlp.Defaults.stop_words  # 326 words

def calculate_jaccard_score(gold_ref, responses):
  jaccard_scores = []

  for gold, response in zip(gold_ref, responses):
      # Convert to sets of words (case-insensitive)
     # gold_words = set(gold.lower().split())
     # resp_words = set(response.lower().split())

      # Filter stop words
      gold_words = set(w.lower() for w in gold.split()
                 if w.lower() not in STOPWORDS and w.isalnum())
      resp_words = set(w.lower() for w in response.split()
                 if w.lower() not in STOPWORDS and w.isalnum())

      # Calculate intersection and union
      intersection = gold_words & resp_words
      union = gold_words | resp_words

      # Jaccard similarity
      if len(union) == 0:
          score = 0.0
      else:
          score = len(intersection) / len(union)

      jaccard_scores.append(score)
  return jaccard_scores





### 2.6 Cosine Similarity

In [ ]:

import re
import numpy as np # Import numpy for reshape
from sklearn.metrics.pairwise import cosine_similarity




def calculate_cosine_similarity(gold_ref, responses):
    """Calculate cosine similarity using local embeddings"""

    # Calculate cosine similarity for each pair
    similarities = []
      # Get embeddings (batch process for speed)
    ref_embeddings = embedding_model.encode(gold_ref)
    resp_embeddings = embedding_model.encode(responses)

    for ref_emb, resp_emb in zip(ref_embeddings, resp_embeddings):
        sim = cosine_similarity([ref_emb], [resp_emb])[0][0]
        similarities.append(sim)

    return similarities






# execute cosine similairty
#rag_cos_sim_scores = compute_cosine_similarity(rag_outputs, ref_qa_pairs, sample_qa_pairs, base_embeddings)
#print("RAG Cosine Similarities:", rag_cos_sim_scores)



### 2.7 RAGAS

| Metric | What It Measures | Requires |
|--------|------------------|----------|
| Faithfulness | Is the answer grounded in the retrieved context? | Question, Answer, Context |
| Answer Relevancy | How relevant is the answer to the question? | Question, Answer |
| Context Precision | Are relevant contexts ranked higher? | Question, Context, Ground Truth |
| Context Recall | Did we retrieve all relevant information? | Context, Ground Truth |
| Context Relevancy | Is the retrieved context relevant to the question? | Question, Context |

In [ ]:
#!pip uninstall ragas
!pip install -q -U ragas
!pip install -q langchain langchain-openai  # or langchain-anthropic

In [ ]:
#openai
from ragas import evaluate
from datasets import Dataset
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision
)
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper


from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from ragas.llms import LangchainLLMWrapper
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch


def calculate_ragas_score_openai(ragas_dataset):

  openai_llm = ChatOpenAI(
      model="gpt-3.5-turbo",
      #api_key=OPENAI_API_KEY,
      temperature=0  # for consistent evaluations
  )

    # Wrap for RAGAS
  ragas_llm = LangchainLLMWrapper(openai_llm)



  #print size of data - number of records
  print(f"Number of records: {len(ragas_dataset)}")

  # Convert to Dataset
  dataset = Dataset.from_dict(ragas_dataset)

  # Evaluate
  result = evaluate(
          dataset,
          metrics=[faithfulness,answer_relevancy,context_precision,context_recall],
          llm=ragas_llm
      )

  ragas_df = result.to_pandas()
  #print(ragas_df)
  # save results to csv file f'{output_path}openai_ragas_results.csv'
  ragas_df.to_csv(f'{output_folder}openai_ragas_results.csv', index=False)
  return result




def calculate_ragas_score_gemma(ragas_dataset):

    # Load Gemma2 model
    model_name = "google/gemma-2-9b-it"  # or "google/gemma-2-2b-it" for smaller

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

    # Create pipeline
    gemma_pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True,
        return_full_text=False
    )

    # Wrap for LangChain
    gemma_llm = HuggingFacePipeline(pipeline=gemma_pipe)

    # Wrap for RAGAS
    ragas_llm = LangchainLLMWrapper(gemma_llm)

    print(f"Number of records: {len(ragas_dataset)}")

    # Convert to Dataset
    dataset = Dataset.from_dict(ragas_dataset)

    # Evaluate
    result = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=ragas_llm
    )

    ragas_df = result.to_pandas()
    ragas_df.to_csv(f'{output_folder}gemma_ragas_results.csv', index=False)
    return result


    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from ragas.llms import LangchainLLMWrapper
import torch

def calculate_ragas_score_gemma_batch(ragas_dataset):

    model_name = "google/gemma-2-9b-it"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token  # Required for batching

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

    # Create pipeline with batching enabled
    gemma_pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True,
        return_full_text=False,
        batch_size=4,  # Enable batching
        padding=True   # Required for batching
    )

    # Wrap for LangChain
    gemma_llm = HuggingFacePipeline(pipeline=gemma_pipe)

    # Wrap for RAGAS
    ragas_llm = LangchainLLMWrapper(gemma_llm)

    print(f"Number of records: {len(ragas_dataset)}")

    dataset = Dataset.from_dict(ragas_dataset)

    result = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=ragas_llm
    )

    return result

/usr/local/lib/python3.12/dist-packages/google/colab/html/_background_server.py:103: DeprecationWarning: make_current is deprecated; start the event loop first
  ioloop.make_current()


### 2.8 SpaCy

In [ ]:
# Download the large English model
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 6.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy


nlp = spacy.load("en_core_web_lg")

def extract_svo_triplets(doc):
    """
    Improved: lemmatize all components and expand dependency patterns
    """
    svo_triplets = []

    for token in doc:
        if token.pos_ in ("VERB", "AUX"):
            subjects = []
            objects = []

            for child in token.children:
                # Expanded subject patterns + lemmatization
                if child.dep_ in ("nsubj", "nsubjpass", "csubj", "csubjpass", "agent"):
                    subjects.append(child.lemma_.lower())

                # Expanded object patterns + lemmatization
                elif child.dep_ in ("dobj", "pobj", "attr", "oprd", "dative", "iobj"):
                    objects.append(child.lemma_.lower())

            verb_lemma = token.lemma_.lower()
            for subj in subjects:
                for obj in objects:
                    svo_triplets.append((subj, verb_lemma, obj))

    return svo_triplets


def semantic_triplet_similarity(triplet1, triplet2):
    """
    Calculate semantic similarity between two SVO triplets using embeddings
    """
    subj1, verb1, obj1 = triplet1
    subj2, verb2, obj2 = triplet2

    doc_subj1, doc_subj2 = nlp(subj1), nlp(subj2)
    doc_verb1, doc_verb2 = nlp(verb1), nlp(verb2)
    doc_obj1, doc_obj2 = nlp(obj1), nlp(obj2)

    subj_sim = doc_subj1.similarity(doc_subj2) if doc_subj1.has_vector and doc_subj2.has_vector else 0
    verb_sim = doc_verb1.similarity(doc_verb2) if doc_verb1.has_vector and doc_verb2.has_vector else 0
    obj_sim = doc_obj1.similarity(doc_obj2) if doc_obj1.has_vector and doc_obj2.has_vector else 0

    # Weighted average (verb is most important)
    return (subj_sim * 0.3 + verb_sim * 0.4 + obj_sim * 0.3)


def calculate_svo_overlap(gold_ref, responses, threshold=0.65):
    """
    Calculate SVO overlap using semantic similarity instead of exact matching
    """
    svo_scores = []

    for ref, response in zip(gold_ref, responses):
        doc_ref = nlp(ref)
        doc_response = nlp(response)

        svo_ref = extract_svo_triplets(doc_ref)
        svo_response = extract_svo_triplets(doc_response)

        if len(svo_ref) == 0 and len(svo_response) == 0:
            score = 1.0
        elif len(svo_ref) == 0 or len(svo_response) == 0:
            score = 0.0
        else:
            # Find semantic matches
            matched_ref = set()
            matched_response = set()

            for i, ref_triplet in enumerate(svo_ref):
                for j, resp_triplet in enumerate(svo_response):
                    sim = semantic_triplet_similarity(ref_triplet, resp_triplet)
                    if sim >= threshold:
                        matched_ref.add(i)
                        matched_response.add(j)

            # Calculate F1-style score
            precision = len(matched_response) / len(svo_response)
            recall = len(matched_ref) / len(svo_ref)

            if precision + recall > 0:
                score = 2 * (precision * recall) / (precision + recall)
            else:
                score = 0.0

        svo_scores.append(score)

    return svo_scores




def calculate_cosine_similarity_spacy( gold_ref, rag_responses):
  """Compute cosine similarity between RAG outputs and reference answers.
  """
  response_cosine_similarities = []
  #embeddings_model = base_embeddings
  """Using spaCy's built-in similarity (cosine similarity)"""
  for ref, resp in zip(gold_ref, rag_responses):
    if resp is None: # Handle cases where there is no response
        response_cosine_similarities.append(0.0) # Assign a default similarity or None
        continue
    doc_ref = nlp(ref)
    doc_resp = nlp(resp)
    # This IS cosine similarity under the hood
    response_cosine_similarities.append(doc_ref.similarity(doc_resp))
  return response_cosine_similarities

In [ ]:
# Check model is loaded
print(f"Model loaded: {nlp}")
print(f"Model has vectors: {nlp.vocab.vectors_length > 0}")

# Test on simple sentence
test_doc = nlp("The cat chased the mouse")
print(f"Tokens: {[token.text for token in test_doc]}")

Model loaded: <spacy.lang.en.English object at 0x7f9ce2f52c60>
Model has vectors: True
Tokens: ['The', 'cat', 'chased', 'the', 'mouse']


In [ ]:
# Test with known simple sentences
test_sentences = [
    "The cat chased the mouse",
    "Python is a language",
    "John loves Python"
]

for sentence in test_sentences:
    doc = nlp(sentence)
    triplets = extract_svo_triplets(doc)
    print(f"'{sentence}' -> {triplets}")

'The cat chased the mouse' -> [('cat', 'chase', 'mouse')]
'Python is a language' -> [('python', 'be', 'language')]
'John loves Python' -> [('john', 'love', 'python')]


### Aggregate Metric (MDS)


We will use an aggremate metric similar to F1 score, calculated as the harmonic mean of the most important metrics for our analysis.
Based on the results, the metrics we care most about for our system are listed in the table below:



| # | Metric | Why Include |
|---|--------|-------------|
| 1 | ragas_score_faithfulness | Measures hallucination - LLM staying grounded in context. Critical for factual QA. |
| 2 | ragas_score_answer_relevancy | Measures if answer actually addresses the question asked. |
| 3 | ragas_score_context_precision | Measures retrieval quality - are retrieved chunks relevant? |
| 4 | ragas_score_context_recall | Measures retrieval coverage - did we find all relevant information? |
| 5 | bert_score_f1 | Measures semantic similarity to gold answer - captures meaning, not just words. |

We define S_{MDS} to be the harmonic mean of 4 metrics, the faithfulness, relevancy, precision and the bert f1 score for semantic meaning. Using the harmonic mean instead of the average allows the impact of each component to reflect in the overall score. Just taking simple average can hide a low subscore if a one of the other subscore is higher. We can assign weights to each of the subscore which we can tune to further improve the performance of our system. Initially, we will keep the weights equal.

In [ ]:
# implement mds as the harmonic mean of the top 5 most important metrics (faithfulness, answer_relevancy, context_precision, bert score f1.)
def calculate_mds(faithfulness, answer_relevancy, context_precision, bert_score_f1):
  epsilon = 1e-10



  # Weights
  weights = [0.25, 0.25, 0.25, 0.25]
  metrics = [
      np.array(faithfulness),
      np.array(answer_relevancy),
      np.array(context_precision),
      np.array(bert_score_f1)
  ]

  # Weighted harmonic mean
  denominator = sum(w / (m + epsilon) for w, m in zip(weights, metrics))
  mds = 1 / denominator

  return mds






### 3.3 The Test Data

You will want to test the system that you (will) have built. Below we give you a validation set that you could take as labeled data (imagine, your user personas would have had these questions and deemed the answers to be good). We also will give you a test set that only contains questions. (This is the set that we will use to get a feel for how well your RAG system corresponds to our Gold model).



In [ ]:
# load gold questions answer  from. validation_questions_answers to csv file f'{output_folder}rag_research_gold.csv'




# read rag_reference_gold from saved file
import pandas as pd
gold_df = pd.read_csv(f'{data_folder}rag_reference_gold.csv')
print(gold_df.shape)
print(gold_df.head())

# read test_questions from saved file
test_df = pd.read_csv(f'{data_folder}test_questions.csv')
print(test_df.shape)
print(test_df.head())



(78, 4)
   question_num                                           question  \
0             0  What defines a large language model in the con...   
1             1  How do large language models like GPT-3 become...   
2             2  What are some of the architectures used in bui...   
3             3  Can you name some notable large language model...   
4             7  What licensing terms are associated with sourc...   

                                gold_answer_research  \
0  A large language model in the context of natur...   
1  Large language models like GPT-3 become capabl...   
2  Some common architectures used in building art...   
3  Some notable large language models include Mis...   
4  Source-available models like Mistral AI's lang...   

                               gold_answer_marketing  
0  A large language model (LLM) is a language mod...  
1  Large language models like GPT-3 become capabl...  
2  The architectures used in building artificial ...  
3  Some notabl

## 4. Tests & Evaluations

Here you should evaluate the results. First, you should implement your evaluation metrics and then you should run evaluation tests. This is really your area, but key results to show are:

1) Your metrics of choice  
2) How  your various models compare to the labeled validation data.

Make sure you look at the results for the marketing team and the research team separately.

**Note:** You do not need to run all models against all labeled questions, as that may take some time. Just do that for a few models/configs, and test a larger set with a smaller subset. But if you use a subset you must justify why you are using that specific subset of questions.

**This is free form so you will need to create your own cells, text documentation as you need, etc.**

After you have implemented you evaluation strategy please answer the questions below in sections 4.1 and 4.2.

Please feel free to add more text and code cells as needed.



### Metrics Candidate
- cosine similarity (semantic)
- length of response
- Jaccquard (word overlap)
- ROUGE (word overlap)
- NDCG. (retrieval) time permitting



| Category | Metrics | Purpose |
|----------|---------|---------|
| RAG-Specific | Context Precision |  Are the retrieved chunks relevant to the question? |
| RAG-Specific | Context Recall |  how much of the retrieved chunks are captured in answer? |
| RAG-Specific | Faithfulness |  Does the answer stick to the retrieved context (no hallucination)? |
| RAG-Specific | Answer Relevancy |  Measures if answer addresses the question |
| Semantic | Cosine Sim (SpaCy) |  Meaning match |
| Semantic | Cosine Similarity |  Meaning match |
| Semantic | BERT F1 |  Contextual match |
| Lexical | ROUGE-L |  Sequence match |
| Lexical | Jaccard |  Set overlap |
| Lexical | BLEU |  N-gram match |
| Semantic | SpaCy Score | Subject Verb Object overlap |

RAG Pipeline Coverage:
├── RETRIEVAL QUALITY
│   ├── Context Precision (relevant chunks?)
│   └── Context Recall (all chunks found?)
│
├── GENERATION QUALITY  
│   ├── Faithfulness (grounded in context?)
│   └── Answer Relevancy (addresses question?)
│
└── GOLD ANSWER COMPARISON
    └── BERT F1 (semantically correct?)



| # | Metric | Why Include |
|---|--------|-------------|
| 1 | ragas_score_faithfulness | Measures hallucination - LLM staying grounded in context. Critical for factual QA. |
| 2 | ragas_score_answer_relevancy | Measures if answer actually addresses the question asked. |
| 3 | ragas_score_context_precision | Measures retrieval quality - are retrieved chunks relevant? |
| 4 | ragas_score_context_recall | Measures retrieval coverage - did we find all relevant information? |
| 5 | bert_score_f1 | Measures semantic similarity to gold answer - captures meaning, not just words. |




| Q# | Faithfulness | | Answer Relevancy | | Context Precision | | Context Recall | | BERT F1 | |
|-----|------|------|------|------|------|------|------|------|------|------|
| | Res. | Mkt. | Res. | Mkt. | Res. | Mkt. | Res. | Mkt. | Res. | Mkt. |
|-----|------|------|------|------|------|------|------|------|------|------|
| Q0 | 0.753 | 0.840 | 0.887 | 0.898 | 0.711 | 0.848 | 0.800 | 1.000 | 0.674 | 0.712 |
| Q7 | 0.273 | 0.333 | 0.371 | 0.369 | 1.000 | 0.507 | 0.800 | 1.000 | 0.557 | 0.617 |
| Q11 | 0.703 | 0.447 | 0.379 | 0.917 | 0.962 | 0.725 | 0.533 | 0.400 | 0.596 | 0.577 |
| Q19 | 0.700 | 0.080 | 0.157 | 0.375 | 0.872 | 0.609 | 0.867 | 0.400 | 0.552 | 0.583 |
| Q27 | 0.784 | 0.853 | 0.976 | 0.905 | 1.000 | 0.918 | 0.750 | 0.900 | 0.680 | 0.692 |
| Q28 | 0.771 | 0.805 | 0.382 | 0.546 | 1.000 | 1.000 | 0.667 | 1.000 | 0.625 | 0.683 |
| Q33 | 0.766 | 0.790 | 0.978 | 0.982 | 1.000 | 1.000 | 0.750 | 1.000 | 0.658 | 0.661 |
| Q53 | 0.770 | 0.577 | 0.710 | 0.857 | 1.000 | 0.961 | 0.700 | 1.000 | 0.595 | 0.652 |
| Q63 | 0.826 | 0.738 | 0.960 | 0.934 | 1.000 | 1.000 | 0.900 | 0.700 | 0.702 | 0.705 |
| Q76 | 0.709 | 0.470 | 0.933 | 0.750 | 1.000 | 0.791 | 1.000 | 1.000 | 0.625 | 0.650 |
| Q87 | 0.603 | 0.510 | 0.921 | 0.924 | 0.881 | 0.430 | 0.880 | 1.000 | 0.678 | 0.659 |
| Q92 | 0.796 | 0.860 | 0.987 | 0.960 | 0.695 | 0.697 | 1.000 | 0.771 | 0.676 | 0.710 |
| Q95 | 0.860 | 0.524 | 0.778 | 0.916 | 0.951 | 0.799 | 0.640 | 0.267 | 0.614 | 0.641 |
| Q103 | 0.623 | 0.640 | 0.947 | 0.964 | 0.973 | 0.936 | 1.000 | 1.000 | 0.692 | 0.764 |
| Q110 | 0.669 | 0.327 | 0.771 | 0.970 | 0.860 | 0.849 | 0.900 | 1.000 | 0.604 | 0.651 |
| **AVG** | **0.707** | **0.586** | **0.742** | **0.818** | **0.927** | **0.808** | **0.812** | **0.829** | **0.635** | **0.664** |

In [ ]:
# Compute each of the metrics defined above and store results to csv file to analyse later






RAG_OUTPUT_FILE = 'rag_final_full_run_mistral_outputs.csv'
#'rag_cohere_full_run_outputs.csv'
#"rag_few_shot_prompts_48_outputs_new.csv"
#"rag_full_validation_questions_outputs.csv"
#'rag_improved_prompts_outputs.csv'
#'rag_prompts_outputs.csv'
# 'rag_retriever_params_outputs_mds.csv'
#'rag_single_question_outputs_1206.csv'
#'rag_llm_params_outputs.csv'
#'rag_embeddings_outputs.csv'
#'rag_retriever_params_outputs.csv'
#'rag_research_textsplitter_outputs.csv'
#'rag_research_baseline_outputs.csv'

EXPERIMENT_NAME = 'full_val_q_mistral'

#

# Prepare RAG output dataframe
output_df, ground_truth_list, ragas_data = create_rag_output_df(RAG_OUTPUT_FILE )


#print size of data - number of records
print(f"Number of records: {output_df.shape}")

#create gold_refs corresponding to the question number in output_df
question_nums = output_df['question_num']
#gold_refs = gold_df[gold_df['question_num'].isin(question_nums)]
#gold_refs = gold_refs[f'gold_answer_{audience_type}'].tolist()
gold_refs = ground_truth_list
print(f"Number of gold references: {len(gold_refs)}")

# Extract cleaned rag responses from output_df
rag_responses = output_df['cleaned_response'].tolist()
ragas_dataset = ragas_data
print(f"Number of cleaned responses: {len(rag_responses)}")

metrics_config = {
    "length_score": {
        'func': calculate_length_ratio_score,
        "args": (gold_refs, rag_responses)
    },
    "rouge_l_score": {
        'func': calculate_rouge_l_score,
        "args": (gold_refs, rag_responses)
    },
    "bleu_score": {
        'func': calculate_bleu_score,
        "args": (gold_refs, rag_responses)
    },
    "bert_score" : {
        'func': calculate_bert_score,
        "args": (gold_refs, rag_responses)  # returns dict with multiple values
    },
    "jaccard_score": {
        'func': calculate_jaccard_score,
        "args": (gold_refs, rag_responses)
    },
    "cosine_similarity": {
        "func": calculate_cosine_similarity,
        "args": (gold_refs, rag_responses)
    },
    "cosine_similarity_spacy": {
        "func": calculate_cosine_similarity_spacy,
        "args": (gold_refs, rag_responses)
    },
    "spacy_score": {
        "func": calculate_svo_overlap,
        "args": (gold_refs, rag_responses)
    },
    "ragas_score": {
        'func': calculate_ragas_score_openai,
        'args': (ragas_dataset,)
    }

}

# Initialize results dictionary
all_results = {"question_num": question_nums}


for metric_name, config in metrics_config.items():
    print(f"Computing {metric_name}...")
    try:
        result = config["func"](*config["args"])

        # Handle different return types
        if isinstance(result, dict):
            # Multi-value metric (like BERTScore)
            for sub_name, value in result.items():
                all_results[f"{metric_name}_{sub_name}"] = value
                print(f" {metric_name}_{sub_name}")
                print(f"Number of {metric_name}_{sub_name} results: {len(value)}")
        elif hasattr(result, 'to_pandas'):
            # RAGAS result
            ragas_df = result.to_pandas()
            for col in ragas_df.columns:
                if col not in ['user_input', 'response', 'retrieved_contexts', 'reference']:
                    all_results[f"{metric_name}_{col}"] = ragas_df[col].values
                    print(f"Number of {metric_name}_{col} results: {len(ragas_df[col].values)}")
        else:
            # Single value metric
            all_results[metric_name] = result
            print(f"Number of {metric_name} results: {len(result)}")
            print(f"{metric_name}")

    except Exception as e:
      print(f"{metric_name} failed: {e}")

# calculate harmonic mean metric mds
mds_result = calculate_mds(all_results['ragas_score_faithfulness'],
                           all_results['ragas_score_answer_relevancy'],
                           all_results['ragas_score_context_precision'],
                           all_results['bert_score_f1'])

all_results['mds'] = mds_result
print(f"MDS: {mds_result}")

# convert all_results to dataframe and write to csv file  f"{output_folder}metric_results.csv"
all_results_df = pd.DataFrame(all_results)
#add custom colmns from experiemnt like chunksize, chunk_overlap
all_results_df['audience_type'] = output_df['audience_type']
all_results_df['experiment_name'] = [EXPERIMENT_NAME] * len(all_results_df)
all_results_df['chunksize'] = output_df['chunksize']
all_results_df['chunkoverlap'] = output_df['chunkoverlap']
all_results_df['embedding_model'] = output_df['embedding_model']
all_results_df['config_name'] = output_df['config_name']
all_results_df['num_retrieve'] = output_df['num_retrieve']
all_results_df['llm_name'] = output_df['llm_name']
all_results_df['llm_temp'] = output_df['llm_temp']
all_results_df['llm_max_new_tokens'] = output_df['llm_max_new_tokens']
all_results_df['llm_top_p'] = output_df['llm_top_p']
all_results_df['llm_do_sample'] = output_df['llm_do_sample']
all_results_df['llm_repetition_penalty'] = output_df['llm_repetition_penalty']
all_results_df['inference_time'] = output_df['inference_time']
all_results_df.to_csv(f"{output_folder}metric_results_{EXPERIMENT_NAME}.csv", index=False)
print(f"Saved metric results to {output_folder}metric_results_{EXPERIMENT_NAME}.csv")





              embedding_model  embedding_dim         config_name  chunksize  \
0  multi-qa-mpnet-base-dot-v1            768  research_prompt_48        500   
1  multi-qa-mpnet-base-dot-v1            768  research_prompt_48        500   
2  multi-qa-mpnet-base-dot-v1            768  research_prompt_48        500   
3  multi-qa-mpnet-base-dot-v1            768  research_prompt_48        500   
4  multi-qa-mpnet-base-dot-v1            768  research_prompt_48        500   

   chunkoverlap  num_retrieve                            llm_name  llm_temp  \
0           100             5  mistralai/Mistral-7B-Instruct-v0.3       0.2   
1           100             5  mistralai/Mistral-7B-Instruct-v0.3       0.2   
2           100             5  mistralai/Mistral-7B-Instruct-v0.3       0.2   
3           100             5  mistralai/Mistral-7B-Instruct-v0.3       0.2   
4           100             5  mistralai/Mistral-7B-Instruct-v0.3       0.2   

   llm_top_p  llm_do_sample  llm_repetition_penalt

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Number of records: 4


/usr/local/lib/python3.12/dist-packages/ragas/evaluation.py:183: DeprecationWarning: Legacy embedding_factory interface is deprecated and will be removed in a future version. Use the modern interface with explicit provider and client parameters: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or import providers directly: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = embedding_factory(


Evaluating:   0%|          | 0/624 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[250]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[254]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[298]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[554]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[562]: TimeoutError()


Number of ragas_score_faithfulness results: 156
Number of ragas_score_answer_relevancy results: 156
Number of ragas_score_context_precision results: 156
Number of ragas_score_context_recall results: 156
MDS: [8.34163098e-01 8.38156272e-01 4.00000000e-10 4.00000000e-10
 4.00000000e-10 4.00000000e-10 2.00000000e-10 4.00000000e-10
 4.00000000e-10 4.00000000e-10 7.13577817e-01 8.57670546e-01
 4.00000000e-10 4.00000000e-10 4.00000000e-10 8.68046883e-01
 8.38339887e-01 4.00000000e-10 6.79576374e-01 8.59970659e-01
 8.92572433e-01 4.00000000e-10 4.00000000e-10 4.00000000e-10
 9.09369741e-01 7.98631554e-01 6.56926545e-01 7.90594094e-01
 6.81708109e-01 8.24967245e-01 6.39863866e-01 8.40329710e-01
 7.79083022e-01 6.60769513e-01 7.02362093e-01 4.00000000e-10
 4.00000000e-10 4.00000000e-10 8.16542639e-01 8.10161897e-01
 7.88674305e-01 8.85905301e-01 8.95405873e-01 8.65484918e-01
 8.87515710e-01 8.22319091e-01 7.45249640e-01 8.41741731e-01
 6.90660877e-01 2.00000000e-10 4.00000000e-10 8.59836836e-01

#### Attempted with gemma2 but got timeout  errors so switched to openAI

In [ ]:
import warnings
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

# Suppress deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

def calculate_ragas_score_gemma(ragas_dataset):

    # =========================================================================
    # 1. LOAD MODEL WITH OPTIMIZATIONS
    # =========================================================================
    model_name = "google/gemma-2-9b-it"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager"  # More stable for long sequences
    )

    # =========================================================================
    # 2. CREATE PIPELINE WITH BETTER SETTINGS
    # =========================================================================
    gemma_pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,      # Reduced from 512 - faster generation
        temperature=0.1,
        do_sample=True,
        return_full_text=False,
        batch_size=1,            # Gemma works better without batching
    )

    # =========================================================================
    # 3. WRAP FOR RAGAS
    # =========================================================================
    gemma_llm = HuggingFacePipeline(pipeline=gemma_pipe)
    ragas_llm = LangchainLLMWrapper(gemma_llm)

    # Use local embeddings (avoid OpenAI default)
    embeddings = HuggingFaceEmbeddings(model_name="multi-qa-mpnet-base-dot-v1")
    ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

    # =========================================================================
    # 4. CONVERT DATASET
    # =========================================================================
    print(f"Number of records: {len(ragas_dataset)}")
    dataset = Dataset.from_dict(ragas_dataset)

    # =========================================================================
    # 5. EVALUATE WITH TIMEOUT FIX
    # =========================================================================
    result = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=False,  # Don't crash on errors
    )

    return result